# 66 — LightRAG : d'où vient l'écart, du graphe ou du tri des passages ?

Le notebook 64 a montré que LightRAG, sur **notre graphe**, retrouve les mêmes
documents que la baseline vectorielle, question par question. L'explication
proposée vient de la lecture de son code : après avoir trouvé les entités,
LightRAG classe les résumés qui leur sont rattachés **par ressemblance
vectorielle avec la question**, avec le même encodeur et le même texte que la
baseline. C'est son réglage par défaut, `kg_chunk_pick_method = VECTOR`.

Ce notebook transforme cette explication en mesure. LightRAG documente un second
réglage, `WEIGHT`, qui choisit les résumés **d'après les entités retrouvées**,
sans les reclasser par ressemblance avec la question. On relance les 20 questions
en ne changeant que ce réglage :

- même graphe (celui du notebook 64, déjà dans le Drive) ;
- mêmes mots-clés (LightRAG les a gardés en cache ; aucun appel au modèle de
  langue n'est autorisé ici, pour que ce soit vérifiable) ;
- mêmes encodages, même matériel.

**Prédiction, posée avant l'exécution.** Si l'explication est juste, les trois
questions single-hop ratées — SH01 (corticosterone), SH03 (prasugrel),
SH07 (sarpogrelate) — doivent retrouver au moins un bon résumé dans les cinq
premiers avec `WEIGHT`. Sinon, l'explication est fausse ou incomplète.

Le notebook affiche aussi, pour chaque question single-hop, **le rang du nœud
attendu** dans le contexte assemblé par LightRAG : cela dit si le nœud est
retrouvé puis son résumé écarté au tri, ou s'il n'est pas retrouvé du tout.

**Entrées** : le dossier `lightrag_notre_graphe/` du Drive (écrit par le notebook 64),
plus `jeu_de_test.json`, `courbe_rappel.json` et `baseline_lightrag.json`
**Sortie** : `baseline_lightrag_weight.json`, à replacer dans `data/processed/`
**Matériel** : GPU T4. Ni Ollama ni qwen : aucune insertion, aucune extraction.

## Cellule 1 — Installation

In [1]:
# Ni Ollama ni qwen : ce notebook ne fait qu'interroger un graphe déjà construit.
!pip install -q lightrag-hku==1.5.7 sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 84.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.3/51.3 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 840.2/840.2 kB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.3/90.3 kB 8.8 MB/s eta 0:00:00


## Cellule 2 — Vérifier le GPU

In [2]:
import torch
print("GPU disponible :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Modèle de GPU  :", torch.cuda.get_device_name(0))

GPU disponible : True
Modèle de GPU  : Tesla T4


## Cellule 3 — Drive et fichiers

In [3]:
import json
import os
import shutil
from google.colab import drive, files

drive.mount("/content/drive")
DOSSIER = "/content/drive/MyDrive/stage-graphrag"
TRAVAIL = f"{DOSSIER}/lightrag_notre_graphe"

ATTENDUS = ["jeu_de_test.json", "courbe_rappel.json", "baseline_lightrag.json"]
manquants = [n for n in ATTENDUS if not os.path.exists(f"{DOSSIER}/{n}")]
if manquants:
    print("À téléverser :", manquants)
    for nom in files.upload():
        shutil.copy(nom, f"{DOSSIER}/{nom}")
for n in ATTENDUS:
    etat = "présent" if os.path.exists(f"{DOSSIER}/{n}") else "MANQUANT"
    print(f"  {n:<26} {etat}")

if os.path.isdir(TRAVAIL):
    print(f"\nDossier LightRAG : présent ({len(os.listdir(TRAVAIL))} fichiers)")
else:
    print("\nDossier LightRAG : MANQUANT — sans lui, il faudrait relancer le notebook 64")

with open(f"{DOSSIER}/jeu_de_test.json", encoding="utf-8") as f:
    questions = json.load(f)
with open(f"{DOSSIER}/courbe_rappel.json", encoding="utf-8") as f:
    GRAPHE = {l["id"]: l["graphe"] for l in json.load(f)["detail"]}
with open(f"{DOSSIER}/baseline_lightrag.json", encoding="utf-8") as f:
    PUBLIE = json.load(f)["classements"]
print(f"{len(questions)} questions chargées ; classements du notebook 64 : {', '.join(PUBLIE)}")

Mounted at /content/drive
  jeu_de_test.json           présent
  courbe_rappel.json         présent
  baseline_lightrag.json     présent

Dossier LightRAG : présent (12 fichiers)
20 questions chargées ; classements du notebook 64 : mix, hybrid


## Cellule 4 — Rouvrir le graphe, sans rien insérer

In [4]:
import lightrag
import networkx as nx
from sentence_transformers import SentenceTransformer
from lightrag import LightRAG, QueryParam
from lightrag.utils import EmbeddingFunc

# Aucun appel au modèle de langue ne doit avoir lieu : les mots-clés des
# questions sont déjà dans le cache écrit par le notebook 64. Si cette fonction
# est appelée, c'est que le cache a été perdu, et la comparaison ne serait plus
# à mots-clés identiques : mieux vaut s'arrêter que mesurer autre chose.
APPELS = []

async def modele(prompt, system_prompt=None, history_messages=[], **kwargs):
    APPELS.append(prompt[:120])
    raise RuntimeError("Mots-clés absents du cache : arrêter et le signaler.")

encodeur = SentenceTransformer("pritamdeka/S-PubMedBert-MS-MARCO", device="cuda")

async def encoder(textes, **kwargs):
    return encodeur.encode(list(textes), normalize_embeddings=True, convert_to_numpy=True)

rag = LightRAG(
    working_dir=TRAVAIL,
    llm_model_func=modele,
    llm_model_name="qwen2.5:7b",   # identité du cache des mots-clés : ne pas changer
    embedding_func=EmbeddingFunc(embedding_dim=768, func=encoder),
    chunk_token_size=4000,
    max_parallel_insert=1,
)
await rag.initialize_storages()

g = nx.read_graphml(f"{TRAVAIL}/graph_chunk_entity_relation.graphml")
print(f"LightRAG {lightrag.__version__} — graphe rouvert : "
      f"{g.number_of_nodes()} entités, {g.number_of_edges()} relations")
print("Attendu (notebook 64)                    : 792 entités, 856 relations")
print("Réglage de départ :", rag.text_chunks.global_config.get("kg_chunk_pick_method"))

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/4.56k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/461k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

INFO: [] Loaded graph from /content/drive/MyDrive/stage-graphrag/lightrag_notre_graphe/graph_chunk_entity_relation.graphml with 792 nodes, 856 edges
INFO: Role LLM Configuration (initialized):
INFO:  - extract: None/None, host=None, max_async=4, timeout=240
INFO:  - keyword: None/None, host=None, max_async=4, timeout=240
INFO:  - query: None/None, host=None, max_async=4, timeout=240
INFO:  - vlm: None/None, host=None, max_async=4, timeout=240
INFO: [] Process 1317 KV load full_docs with 475 records
INFO: [] Process 1317 KV load text_chunks with 475 records
INFO: [] Process 1317 KV load full_entities with 475 records
INFO: [] Process 1317 KV load full_relations with 475 records
INFO: [] Process 1317 KV load entity_chunks with 792 records
INFO: [] Process 1317 KV load relation_chunks with 856 records
INFO: [] Process 1317 KV load llm_response_cache with 1019 records
INFO: [] Process 1317 doc status load doc_status with 475 records


LightRAG 1.5.7 — graphe rouvert : 792 entités, 856 relations
Attendu (notebook 64)                    : 792 entités, 856 relations
Réglage de départ : VECTOR


## Cellule 5 — Les quatre configurations

In [5]:
import logging

MODES = ("mix", "hybrid")
METHODES = ("VECTOR", "WEIGHT")

# L'entité que nomme chaque question single-hop, telle qu'elle figure dans notre
# graphe. Sert à situer le nœud dans le contexte que LightRAG assemble.
ENTITE_ATTENDUE = {"SH01": "corticosterone", "SH02": "terguride", "SH03": "prasugrel",
                   "SH04": "elemene", "SH05": "nilotinib", "SH06": "sb204741",
                   "SH07": "sarpogrelate", "SH08": "capsaicin", "SH09": "azd6140"}

# LightRAG écrit dans son journal la façon dont il a choisi les passages :
# on la relève, plutôt que de supposer que le réglage a bien été pris en compte.
JOURNAL = []

class Capteur(logging.Handler):
    def emit(self, enregistrement):
        message = enregistrement.getMessage()
        if "-related chunks by" in message:
            JOURNAL.append(message)

logging.getLogger("lightrag").addHandler(Capteur())

CLASSEMENTS, RANGS_ENTITE = {}, {}
for methode in METHODES:
    rag.text_chunks.global_config["kg_chunk_pick_method"] = methode
    for mode in MODES:
        cle = f"{methode} {mode}"
        CLASSEMENTS[cle], RANGS_ENTITE[cle] = {}, {}
        debut = len(JOURNAL)
        for q in questions:
            r = await rag.aquery_data(q["question"], param=QueryParam(mode=mode, enable_rerank=False))
            donnees = r.get("data", {})
            vus = []
            for c in donnees.get("chunks", []):
                pm = str(c.get("file_path", ""))
                if pm and pm not in vus:
                    vus.append(pm)
            CLASSEMENTS[cle][q["id"]] = vus
            attendue = ENTITE_ATTENDUE.get(q["id"])
            if attendue:
                noms = [str(e.get("entity") or e.get("entity_name") or "").lower()
                        for e in donnees.get("entities", [])]
                rangs = [i + 1 for i, n in enumerate(noms) if attendue in n]
                RANGS_ENTITE[cle][q["id"]] = rangs[0] if rangs else None
        traces = {"ressemblance vectorielle" if "vector similarity" in m else "poids des entités"
                  for m in JOURNAL[debut:]}
        print(f"{cle:<15} {len(CLASSEMENTS[cle])} questions   "
              f"passages choisis par : {', '.join(sorted(traces)) or 'aucune trace'}")

print("\nAppels au modèle de langue :", len(APPELS), "— doit être 0 (mots-clés pris dans le cache)")

INFO: Embedding func: 8 new workers initialized (Timeouts: Func: 30s, Worker: 60s, Health Check: 75s)
INFO: Query edges: pharmacokinetic ratio, enzyme, paroxetine (top_k:40, cosine:0.2)
INFO: Global query: 40 entites, 40 relations
INFO: Naive query: 20 chunks (chunk_top_k:20 cosine:0.2)
INFO: Raw search results: 40 entities, 40 relations, 20 vector chunks
INFO: After truncation: 40 entities, 40 relations
INFO: Selecting 92 from 92 entity-related chunks by vector similarity
INFO: Find no additional relations-related chunks from 40 relations
INFO: Round-robin merged chunks: 112 -> 100 (deduplicated 12)
INFO: Final context: 40 entities, 40 relations, 20 chunks
INFO: Final chunks S+F/O: E2/1 C1/2 E1/2 C1/3 E3/3 C1/4 E1/4 E14/5 E3/6 E2/7 C1/8 E2/8 E2/9 E2/10 E1/11 C1/12 E2/12 E1/13 C1/14 E1/14
INFO: Query edges: signalling pathway, platelet receptor, cangrelor antagonizes (top_k:40, cosine:0.2)
INFO: Global query: 45 entites, 40 relations
INFO: Naive query: 20 chunks (chunk_top_k:20 cosine:

VECTOR mix      20 questions   passages choisis par : ressemblance vectorielle


INFO: Query nodes: amiloride inhibition (top_k:40, cosine:0.2)
INFO: Local query: 40 entites, 209 relations
INFO: Query edges: muscle-development process, protease activation (top_k:40, cosine:0.2)
INFO: Global query: 62 entites, 40 relations
INFO: Raw search results: 96 entities, 234 relations, 0 vector chunks
INFO: After truncation: 81 entities, 214 relations
INFO: Selecting 121 from 121 entity-related chunks by vector similarity
INFO: Find 3 additional chunks in 3 relations (deduplicated 114)
INFO: Selecting 3 from 3 relation-related chunks by vector similarity
INFO: Round-robin merged chunks: 124 -> 124 (deduplicated 0)
INFO: Final context: 81 entities, 214 relations, 20 chunks
INFO: Final chunks S+F/O: E8/1 R1/1 E4/2 R2/2 E4/3 R1/3 E2/4 E9/5 E2/6 E2/7 E1/8 E1/9 E1/10 E3/11 E1/12 E2/13 E2/14 E2/15 E1/16 E2/17
INFO: Query nodes: ergotamine (top_k:40, cosine:0.2)
INFO: Local query: 40 entites, 286 relations
INFO: Query edges: ergot derivative, agonist, 5-HT2B receptors (top_k:40, cos

VECTOR hybrid   20 questions   passages choisis par : ressemblance vectorielle


INFO: After truncation: 45 entities, 40 relations
INFO: Selecting 64 from 64 entity-related chunks by weighted polling
INFO: Find no additional relations-related chunks from 40 relations
INFO: Round-robin merged chunks: 84 -> 66 (deduplicated 18)
INFO: Final context: 45 entities, 40 relations, 20 chunks
INFO: Final chunks S+F/O: E6/2 E9/1 E2/4 E2/3 E2/30 E2/28 E2/5 E2/13 E6/6 C1/7 E4/7 E2/29 E2/8 E2/9 E2/32 E3/10 E1/37 E3/11 E2/15 E3/12
INFO: Query nodes: amiloride inhibition (top_k:40, cosine:0.2)
INFO: Local query: 40 entites, 209 relations
INFO: Query edges: muscle-development process, protease activation (top_k:40, cosine:0.2)
INFO: Global query: 62 entites, 40 relations
INFO: Naive query: 20 chunks (chunk_top_k:20 cosine:0.2)
INFO: Raw search results: 96 entities, 234 relations, 20 vector chunks
INFO: After truncation: 81 entities, 214 relations
INFO: Selecting 121 from 121 entity-related chunks by weighted polling
INFO: Find 3 additional chunks in 3 relations (deduplicated 114)
I

WEIGHT mix      20 questions   passages choisis par : poids des entités


INFO: Local query: 40 entites, 209 relations
INFO: Query edges: muscle-development process, protease activation (top_k:40, cosine:0.2)
INFO: Global query: 62 entites, 40 relations
INFO: Raw search results: 96 entities, 234 relations, 0 vector chunks
INFO: After truncation: 81 entities, 214 relations
INFO: Selecting 121 from 121 entity-related chunks by weighted polling
INFO: Find 3 additional chunks in 3 relations (deduplicated 114)
INFO: Selecting 3 from 3 relation-related chunks by weighted polling
INFO: Round-robin merged chunks: 124 -> 124 (deduplicated 0)
INFO: Final context: 81 entities, 214 relations, 20 chunks
INFO: Final chunks S+F/O: E8/1 R1/1 E9/2 R2/2 E3/3 R1/3 E1/4 E1/5 E1/6 E2/7 E1/8 E1/9 E1/10 E1/11 E2/12 E4/13 E2/14 E2/15 E1/16 E1/17
INFO: Query nodes: ergotamine (top_k:40, cosine:0.2)
INFO: Local query: 40 entites, 286 relations
INFO: Query edges: ergot derivative, agonist, 5-HT2B receptors (top_k:40, cosine:0.2)
INFO: Global query: 53 entites, 40 relations
INFO: Raw s

WEIGHT hybrid   20 questions   passages choisis par : poids des entités

Appels au modèle de langue : 0 — doit être 0 (mots-clés pris dans le cache)


## Cellule 6 — Résultats

In [6]:
BUDGETS = (1, 3, 5, 10, 20)
COLONNES = [(cle, CLASSEMENTS[cle]) for cle in CLASSEMENTS] + [("notre recherche", GRAPHE)]

def rappel(classement, K, categorie=None):
    qs = [q for q in questions if categorie is None or q["type"] == categorie]
    return sum(len(set(q["pmids"]) & set(classement[q["id"]][:K])) / len(q["pmids"])
               for q in qs) / len(qs)

print("CONTRÔLE — le réglage VECTOR doit reproduire les classements du notebook 64")
for mode in MODES:
    a, b = CLASSEMENTS[f"VECTOR {mode}"], PUBLIE[mode]
    differentes = [q["id"] for q in questions if a[q["id"]] != b[q["id"]]]
    print(f"  {mode:<7} " + ("IDENTIQUE" if not differentes
                             else "DIFFÉRENT sur " + ", ".join(differentes)))

print("\nRAPPEL SELON LE BUDGET DOCUMENTAIRE")
print(f"{'K':>12}" + "".join(f"{nom:>17}" for nom, _ in COLONNES))
for K in BUDGETS:
    print(f"{K:>12}" + "".join(f"{rappel(c, K):>17.2f}" for _, c in COLONNES))

print("\nRAPPEL@5 PAR CATÉGORIE")
for cat in ("single-hop", "pont", "intersection"):
    print(f"{cat:>12}" + "".join(f"{rappel(c, 5, cat):>17.2f}" for _, c in COLONNES))

print("\nLA PRÉDICTION — rappel@5 des trois questions que la vectorielle rate")
print(f"{'question':>12}" + "".join(f"{nom:>17}" for nom, _ in COLONNES))
for qid in ("SH01", "SH03", "SH07"):
    q = [x for x in questions if x["id"] == qid][0]
    print(f"{qid:>12}" + "".join(
        f"{len(set(q['pmids']) & set(c[qid][:5])) / len(q['pmids']):>17.2f}" for _, c in COLONNES))

print("\nRANG DU NŒUD ATTENDU dans le contexte assemblé par LightRAG")
print(f"{'question':>12}" + "".join(f"{cle:>17}" for cle in CLASSEMENTS))
for qid in sorted(ENTITE_ATTENDUE):
    print(f"{qid:>12}" + "".join(
        f"{(RANGS_ENTITE[cle].get(qid) or 'absent'):>17}" for cle in CLASSEMENTS))

CONTRÔLE — le réglage VECTOR doit reproduire les classements du notebook 64
  mix     IDENTIQUE
  hybrid  IDENTIQUE

RAPPEL SELON LE BUDGET DOCUMENTAIRE
           K       VECTOR mix    VECTOR hybrid       WEIGHT mix    WEIGHT hybrid  notre recherche
           1             0.38             0.40             0.38             0.55             0.45
           3             0.53             0.53             0.72             0.74             0.78
           5             0.55             0.57             0.82             0.88             0.83
          10             0.59             0.68             0.93             0.93             0.88
          20             0.76             0.80             0.93             0.96             0.95

RAPPEL@5 PAR CATÉGORIE
  single-hop             0.67             0.67             0.94             0.89             1.00
        pont             0.50             0.50             0.67             0.67             0.67
intersection             0.44          

## Cellule 7 — Enregistrer et télécharger

In [7]:
resultat = {"lightrag": lightrag.__version__, "modele": "qwen2.5:7b",
            "appels_modele_de_langue": len(APPELS),
            "entite_attendue": ENTITE_ATTENDUE,
            "classements": CLASSEMENTS, "rangs_entite": RANGS_ENTITE}
with open("baseline_lightrag_weight.json", "w", encoding="utf-8") as f:
    json.dump(resultat, f, ensure_ascii=False, indent=1)
shutil.copy("baseline_lightrag_weight.json", f"{DOSSIER}/baseline_lightrag_weight.json")
files.download("baseline_lightrag_weight.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>